In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

In [ ]:
data_path = "../data/processed/mieszkania_clean.csv"

df = pd.read_csv(data_path)

if "year_month" in df.columns:
    df["year_month"] = pd.to_datetime(df["year_month"])

print("Liczba rekordów:", df.shape[0])
print("Liczba kolumn:", df.shape[1])

df.head()

In [ ]:
print("Dostępne kolumny:")
print(df.columns.tolist())

In [ ]:
df_model = df.copy()

if "id" in df_model.columns and "year_month" in df_model.columns:
    df_model = df_model.sort_values("year_month").drop_duplicates("id", keep="last")
elif "id" in df_model.columns:
    df_model = df_model.drop_duplicates("id", keep="last")

print("Liczba rekordów po usunięciu powtarzających się ofert:", df_model.shape[0])

In [ ]:
features = [
    "city",
    "squareMeters",
    "rooms",
    "floor",
    "floorCount",
    "buildYear",
    "centreDistance",
    "poiCount"
]

target = "price"

missing_cols = [col for col in features + [target] if col not in df_model.columns]

if missing_cols:
    print("Brakuje kolumn:", missing_cols)
else:
    print("Wszystkie potrzebne kolumny są dostępne.")

In [ ]:
df_model = df_model.dropna(subset=features + [target]).copy()

print("Liczba rekordów po usunięciu braków w kolumnach modelowych:", df_model.shape[0])

X = df_model[features]
y = df_model[target]

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Liczba rekordów treningowych:", X_train.shape[0])
print("Liczba rekordów testowych:", X_test.shape[0])

In [ ]:
numeric_features = [
    "squareMeters",
    "rooms",
    "floor",
    "floorCount",
    "buildYear",
    "centreDistance",
    "poiCount"
]

categorical_features = ["city"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror"
        ))
    ]
)

model

In [ ]:
model.fit(X_train, y_train)

print("Model został wytrenowany.")

In [ ]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Wyniki modelu:")
print(f"R²: {r2:.4f}")
print(f"MAE: {mae:,.2f} zł")
print(f"RMSE: {rmse:,.2f} zł")

In [ ]:
results = X_test.copy()
results["cena_rzeczywista"] = y_test.values
results["cena_przewidywana"] = y_pred
results["roznica"] = results["cena_rzeczywista"] - results["cena_przewidywana"]
results["roznica_proc"] = results["roznica"] / results["cena_przewidywana"] * 100

results.head(10)

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred, alpha=0.3)

min_price = min(y_test.min(), y_pred.min())
max_price = max(y_test.max(), y_pred.max())

plt.plot([min_price, max_price], [min_price, max_price])

plt.title("Cena rzeczywista vs cena przewidywana")
plt.xlabel("Cena rzeczywista [zł]")
plt.ylabel("Cena przewidywana [zł]")

plt.tight_layout()

os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/model_cena_rzeczywista_vs_przewidywana.png", dpi=300)

plt.show()

In [ ]:
def ocen_oferte(cena_ofertowa, cena_modelowa):
    roznica_proc = (cena_ofertowa - cena_modelowa) / cena_modelowa * 100

    if roznica_proc < -10:
        ocena = "okazja cenowa"
    elif roznica_proc <= 10:
        ocena = "cena rynkowa"
    else:
        ocena = "oferta droga"

    return roznica_proc, ocena

In [ ]:
sample_offer = X_test.iloc[[0]]
real_price = y_test.iloc[0]

predicted_price = model.predict(sample_offer)[0]

roznica_proc, ocena = ocen_oferte(real_price, predicted_price)

print("Przykładowa oferta:")
display(sample_offer)

print(f"Cena rzeczywista: {real_price:,.2f} zł")
print(f"Cena przewidywana przez model: {predicted_price:,.2f} zł")
print(f"Różnica procentowa: {roznica_proc:.2f}%")
print(f"Ocena oferty: {ocena}")

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

results_path = "../data/processed/wyniki_modelu.csv"

results.to_csv(results_path, index=False)

print(f"Wyniki zapisane w: {results_path}")

In [ ]:
summary = pd.DataFrame({
    "Metryka": ["R²", "MAE", "RMSE", "Liczba obserwacji w modelu"],
    "Wartość": [
        round(r2, 4),
        round(mae, 2),
        round(rmse, 2),
        len(df_model)
    ]
})

summary 